In [1]:
import os, glob, json, math, random, hashlib, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt

SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 20
LR = 2e-4
NUM_WORKERS = 2
PATIENCE = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything()
print("Device:", DEVICE)
print("Kaggle inputs:", os.listdir("/kaggle/input") if Path("/kaggle/input").exists() else "Not on Kaggle")

Device: cuda
Kaggle inputs: ['datasets']


In [5]:

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(
    "/kaggle/input/datasets/ozlemhakdagli/"
    "retinal-fundus-multi-disease-image-dataset-rfmid"
)

IMAGE_EXTS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

DATASETS = [
    {
        "name": "RFMiD",
        "partition": "train",
        "csv": ROOT / "Training_set/RFMiD_Training_Labels.csv",
        "image_root": ROOT / "Training_set",
    },
    {
        "name": "RFMiD",
        "partition": "val",
        "csv": ROOT / "Validation_set/RFMiD_Validation_Labels.csv",
        "image_root": ROOT / "Validation_set",
    },
    {
        "name": "RFMiD",
        "partition": "test",
        "csv": ROOT / "Test_set/RFMiD_Testing_Labels.csv",
        "image_root": ROOT / "Test_set",
    },
]


def normalize_id(value):
    if pd.isna(value):
        return None

    if isinstance(value, (int, np.integer)):
        return str(int(value))

    if isinstance(value, (float, np.floating)):
        if float(value).is_integer():
            return str(int(value))

    value = Path(str(value).strip()).stem

    if value.isdigit():
        return str(int(value))

    return value.lower()


def to_binary_ode(value):
    if pd.isna(value):
        return np.nan

    if isinstance(value, (int, float, np.integer, np.floating)):
        if float(value) in (0.0, 1.0):
            return int(value)

    value = str(value).strip().lower()

    if value in {"1", "true", "yes", "positive", "present"}:
        return 1

    if value in {"0", "false", "no", "negative", "absent"}:
        return 0

    raise ValueError(f"Unexpected ODE label: {value!r}")



rows = []
missing = []
ambiguous = []

for cfg in DATASETS:

    csv_path = cfg["csv"]
    image_root = cfg["image_root"]
    partition = cfg["partition"]

    print(f"\nProcessing {partition.upper()}...")

    if not csv_path.exists():
        raise FileNotFoundError(csv_path)

    if not image_root.exists():
        raise FileNotFoundError(image_root)

    df = pd.read_csv(csv_path)

    required = {"ID", "ODE"}
    if not required.issubset(df.columns):
        raise ValueError(
            f"Expected columns {required}, got {list(df.columns)}"
        )

    # Build an index ONLY from this partition's image directory.
    image_index = {}

    for p in image_root.rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            key = normalize_id(p.stem)
            image_index.setdefault(key, []).append(p)

    print("CSV rows:", len(df))
    print("Images indexed:", sum(len(v) for v in image_index.values()))

    matched = 0

    for _, r in df.iterrows():

        image_id = normalize_id(r["ID"])

        if image_id is None:
            continue

        candidates = image_index.get(image_id, [])

        if len(candidates) == 0:
            missing.append((partition, r["ID"]))
            continue

        if len(candidates) > 1:
            ambiguous.append((partition, r["ID"], candidates))
            continue

        label = to_binary_ode(r["ODE"])

        if pd.isna(label):
            continue

        rows.append({
            "image_path": str(candidates[0]),
            "label": int(label),
            "source": cfg["name"],
            "partition": partition,
            "image_id": str(r["ID"])
        })

        matched += 1

    print("Matched:", matched)


manifest = pd.DataFrame(rows)

if manifest.empty:
    raise RuntimeError("No images were matched.")

manifest = manifest.drop_duplicates(
    subset=["image_path"]
).reset_index(drop=True)

print("\n" + "=" * 55)
print("RFMiD 1.0 DATASET SUMMARY")
print("=" * 55)

print("Total matched images:", len(manifest))
print("Missing image IDs:", len(missing))
print("Ambiguous image IDs:", len(ambiguous))

print("\nPartition distribution:")
print(manifest["partition"].value_counts())

print("\nODE label distribution:")
display(
    manifest.groupby(["partition", "label"])
    .size()
    .rename("count")
    .reset_index()
)

print("\nPartition and class counts:")
display(
    manifest.groupby(["source", "partition", "label"])
    .size()
    .rename("count")
    .reset_index()
)

expected = {"train", "val", "test"}
actual = set(manifest["partition"].unique())

if actual != expected:
    raise RuntimeError(
        f"Expected partitions {expected}, but found {actual}"
    )

if ambiguous:
    raise RuntimeError(
        f"{len(ambiguous)} ambiguous IDs remain. "
        "Inspect them before training."
    )

print("\nRFMiD 1.0 manifest is ready.")


Processing TRAIN...
CSV rows: 1920
Images indexed: 1920
Matched: 1920

Processing VAL...
CSV rows: 640
Images indexed: 640
Matched: 640

Processing TEST...
CSV rows: 640
Images indexed: 640
Matched: 640

RFMiD 1.0 DATASET SUMMARY
Total matched images: 3200
Missing image IDs: 0
Ambiguous image IDs: 0

Partition distribution:
partition
train    1920
val       640
test      640
Name: count, dtype: int64

ODE label distribution:


,partition,label,count
0,test,0,623
1,test,1,17
2,train,0,1862
3,train,1,58
4,val,0,619
5,val,1,21



Partition and class counts:


,source,partition,label,count
0,RFMiD,test,0,623
1,RFMiD,test,1,17
2,RFMiD,train,0,1862
3,RFMiD,train,1,58
4,RFMiD,val,0,619
5,RFMiD,val,1,21



RFMiD 1.0 manifest is ready.


In [6]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

manifest["sha256"] = manifest["image_path"].map(sha256_file)
priority = {"test": 0, "val": 1, "train": 2}
manifest["priority"] = manifest["partition"].map(priority)
manifest = manifest.sort_values(["sha256", "priority"]).drop_duplicates("sha256", keep="first")
manifest = manifest.drop(columns=["priority"]).reset_index(drop=True)

print("After exact duplicate removal:", len(manifest))
display(manifest.groupby(["source", "partition", "label"]).size().rename("n").reset_index())

for part in ["train", "val", "test"]:
    sub = manifest[manifest.partition == part]
    if sub.empty:
        raise ValueError(f"No {part} samples. You need explicit official {part} partition(s).")
    counts = sub.label.value_counts().to_dict()
    print(part, counts)
    if set(counts) != {0, 1}:
        raise ValueError(f"{part} must contain both ODE=0 and ODE=1. Counts={counts}")

# Save an audit manifest in the Kaggle working directory.
manifest.to_csv("/kaggle/working/papilalert_manifest.csv", index=False)

After exact duplicate removal: 3200


,source,partition,label,n
0,RFMiD,test,0,623
1,RFMiD,test,1,17
2,RFMiD,train,0,1862
3,RFMiD,train,1,58
4,RFMiD,val,0,619
5,RFMiD,val,1,21


train {0: 1862, 1: 58}
val {0: 619, 1: 21}
test {0: 623, 1: 17}


In [7]:
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

train_tfms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.90, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(5),
    transforms.ColorJitter(brightness=0.05, contrast=0.05, saturation=0.03, hue=0.005),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tfms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class FundusODE(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row = self.frame.iloc[i]
        with Image.open(row.image_path) as im:
            x = self.transform(im.convert("RGB"))
        return x, torch.tensor(int(row.label), dtype=torch.long)

train_df = manifest[manifest.partition == "train"].copy()
val_df = manifest[manifest.partition == "val"].copy()
test_df = manifest[manifest.partition == "test"].copy()

train_ds = FundusODE(train_df, train_tfms)
val_ds = FundusODE(val_df, eval_tfms)
test_ds = FundusODE(test_df, eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

In [8]:
weights = models.MobileNet_V3_Small_Weights.DEFAULT
model = models.mobilenet_v3_small(weights=weights)
in_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_features, 2)
model = model.to(DEVICE)

counts = train_df.label.value_counts().sort_index()
class_weights = torch.tensor([
    len(train_df) / (2 * max(int(counts.get(0, 0)), 1)),
    len(train_df) / (2 * max(int(counts.get(1, 0)), 1))
], dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)
print("Class weights [no ODE, ODE]:", class_weights.detach().cpu().numpy())

Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth


100%|██████████| 9.83M/9.83M [00:00<00:00, 116MB/s]


Class weights [no ODE, ODE]: [ 0.51557463 16.551723  ]


In [9]:
from sklearn.metrics import roc_auc_score

def epoch_pass(loader, training):
    model.train(training)
    losses, ys, probs = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(xb)
            loss = criterion(logits, yb)
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                optimizer.step()
        losses.append(float(loss.item()) * len(yb))
        ys.extend(yb.detach().cpu().numpy().tolist())
        probs.extend(torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy().tolist())
    y = np.asarray(ys, dtype=int); p = np.asarray(probs)
    auc = roc_auc_score(y, p) if len(np.unique(y)) == 2 else float("nan")
    pred = (p >= 0.5).astype(int)
    return {
        "loss": float(sum(losses) / max(len(y), 1)),
        "auc": float(auc),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "sensitivity_at_0.5": float(recall_score(y, pred, zero_division=0)),
        "specificity_at_0.5": float(recall_score(1-y, 1-pred, zero_division=0)),
        "y": y, "p": p
    }

OUT = Path("/kaggle/working/papilalert_run")
OUT.mkdir(parents=True, exist_ok=True)
best_auc, best_epoch, stale = -1, 0, 0
history = []
best_path = OUT / "best_checkpoint.pt"

for epoch in range(1, EPOCHS + 1):
    tr = epoch_pass(train_loader, True)
    va = epoch_pass(val_loader, False)
    score = va["auc"] if np.isfinite(va["auc"]) else va["macro_f1"]
    scheduler.step(score)
    history.append({
        "epoch": epoch, "train_loss": tr["loss"], "train_auc": tr["auc"],
        "val_loss": va["loss"], "val_auc": va["auc"], "val_macro_f1": va["macro_f1"]
    })
    print(f"Epoch {epoch:02d} | train loss {tr['loss']:.4f} AUC {tr['auc']:.4f} "
          f"| val loss {va['loss']:.4f} AUC {va['auc']:.4f} F1 {va['macro_f1']:.4f}")
    if score > best_auc:
        best_auc, best_epoch, stale = score, epoch, 0
        torch.save({
            "model_state_dict": model.state_dict(),
            "classes": ["no_ode", "optic_disc_edema"],
            "image_size": IMAGE_SIZE,
            "metadata": {
                "model_name": "MobileNetV3-Small",
                "model_version": "0.2-ode-research",
                "target": "optic_disc_edema_not_papilledema_diagnosis",
                "trained_at_utc": pd.Timestamp.utcnow().isoformat(),
                "best_validation_auc": float(va["auc"]),
                "best_epoch": epoch,
                "seed": SEED,
                "training_sources": sorted(manifest[manifest.partition == "train"].source.unique().tolist()),
                "clinical_status": "NOT CLINICALLY VALIDATED",
                "warning": "ODE is not synonymous with papilledema. Research-only; not for diagnosis or care decisions."
            },
            # Conservative prototype abstention thresholds; these are not clinically calibrated.
            "abstain_score_threshold": 0.60,
            "abstain_margin_threshold": 0.15,
        }, best_path)
    else:
        stale += 1
    if stale >= PATIENCE:
        print("Early stopping.")
        break

pd.DataFrame(history).to_csv(OUT / "history.csv", index=False)
print("Best epoch:", best_epoch, "best val AUROC:", best_auc)
print("Checkpoint:", best_path)

Epoch 01 | train loss 0.5909 AUC 0.7975 | val loss 1.1919 AUC 0.8272 F1 0.5375
Epoch 02 | train loss 0.3071 AUC 0.9625 | val loss 2.1321 AUC 0.8303 F1 0.4917
Epoch 03 | train loss 0.2784 AUC 0.9779 | val loss 0.9461 AUC 0.9341 F1 0.7583
Epoch 04 | train loss 0.2517 AUC 0.9749 | val loss 0.8590 AUC 0.9528 F1 0.8187
Epoch 05 | train loss 0.1379 AUC 0.9914 | val loss 1.0528 AUC 0.9445 F1 0.8289
Epoch 06 | train loss 0.1314 AUC 0.9952 | val loss 1.1877 AUC 0.9441 F1 0.7436
Epoch 07 | train loss 0.0979 AUC 0.9974 | val loss 0.3756 AUC 0.9667 F1 0.7275
Epoch 08 | train loss 0.0832 AUC 0.9980 | val loss 0.6711 AUC 0.9770 F1 0.8810
Epoch 09 | train loss 0.0818 AUC 0.9976 | val loss 0.9411 AUC 0.9806 F1 0.8289
Epoch 10 | train loss 0.0902 AUC 0.9982 | val loss 0.5203 AUC 0.9782 F1 0.8273
Epoch 11 | train loss 0.0684 AUC 0.9990 | val loss 1.6630 AUC 0.9475 F1 0.7851
Epoch 12 | train loss 0.0236 AUC 0.9999 | val loss 1.4871 AUC 0.9622 F1 0.8077
Epoch 13 | train loss 0.0046 AUC 1.0000 | val loss 2

In [10]:
checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
te = epoch_pass(test_loader, False)
y_test, p_test = te["y"], te["p"]
pred_test = (p_test >= 0.5).astype(int)

print("TEST AUROC:", roc_auc_score(y_test, p_test))
print("TEST AUPRC:", average_precision_score(y_test, p_test))
print(classification_report(y_test, pred_test, target_names=["no_ode", "optic_disc_edema"], zero_division=0))
print("Confusion matrix [rows=true, cols=pred]:")
print(confusion_matrix(y_test, pred_test))

# Source-wise test metrics reveal whether one source dominates performance.
test_results = test_df[["image_path", "source", "label"]].copy().reset_index(drop=True)
test_results["ode_score"] = p_test
test_results["prediction_at_0.5"] = pred_test
for source, g in test_results.groupby("source"):
    if g.label.nunique() == 2:
        print(source, "n=", len(g), "AUROC=", roc_auc_score(g.label, g.ode_score),
              "sensitivity=", recall_score(g.label, g["prediction_at_0.5"], zero_division=0))
    else:
        print(source, "n=", len(g), "only one class; AUROC undefined")
test_results.to_csv(OUT / "test_predictions.csv", index=False)

TEST AUROC: 0.9671419129449532
TEST AUPRC: 0.784927031048735
                  precision    recall  f1-score   support

          no_ode       0.99      1.00      1.00       623
optic_disc_edema       0.92      0.71      0.80        17

        accuracy                           0.99       640
       macro avg       0.96      0.85      0.90       640
    weighted avg       0.99      0.99      0.99       640

Confusion matrix [rows=true, cols=pred]:
[[622   1]
 [  5  12]]
RFMiD n= 640 AUROC= 0.9671419129449532 sensitivity= 0.7058823529411765


In [11]:
# Kaggle output artifact
shutil_copy = __import__("shutil").copy2
shutil_copy(best_path, "/kaggle/working/papilalert_mobilenet_v3_small.pt")
with open("/kaggle/working/training_metadata.json", "w") as f:
    json.dump({
        "target": "optic_disc_edema",
        "not_equivalent_to_papilledema": True,
        "best_epoch": best_epoch,
        "best_validation_auc": float(best_auc),
        "test_auc": float(roc_auc_score(y_test, p_test)),
        "test_auprc": float(average_precision_score(y_test, p_test)),
        "split_policy": "official source partitions; exact duplicate hash priority test>val>train",
        "clinical_validation": "not performed"
    }, f, indent=2)
print("Ready to download: /kaggle/working/papilalert_mobilenet_v3_small.pt")

Ready to download: /kaggle/working/papilalert_mobilenet_v3_small.pt


In [14]:
import torch
import os

paths = [
    "/kaggle/working/papilalert_run/best_checkpoint.pt",
    "/kaggle/working/papilalert_mobilenet_v3_small.pt"
]

for path in paths:

    print("\n" + "=" * 60)
    print(path)

    if not os.path.exists(path):
        print("FILE DOES NOT EXIST")
        continue

    ckpt = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Checkpoint type:", type(ckpt))

    if isinstance(ckpt, dict):
        print("Keys:", list(ckpt.keys()))
        print("Classes:", ckpt.get("classes"))
        print("Epoch:", ckpt.get("epoch"))
        print("Best validation AUC:", ckpt.get("best_val_auc"))


/kaggle/working/papilalert_run/best_checkpoint.pt
Checkpoint type: <class 'dict'>
Keys: ['model_state_dict', 'classes', 'image_size', 'metadata', 'abstain_score_threshold', 'abstain_margin_threshold']
Classes: ['no_ode', 'optic_disc_edema']
Epoch: None
Best validation AUC: None

/kaggle/working/papilalert_mobilenet_v3_small.pt
Checkpoint type: <class 'dict'>
Keys: ['model_state_dict', 'classes', 'image_size', 'metadata', 'abstain_score_threshold', 'abstain_margin_threshold']
Classes: ['no_ode', 'optic_disc_edema']
Epoch: None
Best validation AUC: None


In [15]:
import torch

best = torch.load(
    "/kaggle/working/papilalert_run/best_checkpoint.pt",
    map_location="cpu",
    weights_only=False
)

exported = torch.load(
    "/kaggle/working/papilalert_mobilenet_v3_small.pt",
    map_location="cpu",
    weights_only=False
)

print("Best metadata:", best.get("metadata"))
print("Exported metadata:", exported.get("metadata"))

same = all(
    torch.equal(best["model_state_dict"][k],
                exported["model_state_dict"][k])
    for k in best["model_state_dict"]
)

print("Are model weights identical?", same)

Best metadata: {'model_name': 'MobileNetV3-Small', 'model_version': '0.2-ode-research', 'target': 'optic_disc_edema_not_papilledema_diagnosis', 'trained_at_utc': '2026-11-17T16:05:02.760871+00:00', 'best_validation_auc': 0.9806138933764135, 'best_epoch': 9, 'seed': 42, 'training_sources': ['RFMiD'], 'clinical_status': 'NOT CLINICALLY VALIDATED', 'warning': 'ODE is not synonymous with papilledema. Research-only; not for diagnosis or care decisions.'}
Exported metadata: {'model_name': 'MobileNetV3-Small', 'model_version': '0.2-ode-research', 'target': 'optic_disc_edema_not_papilledema_diagnosis', 'trained_at_utc': '2026-11-17T16:05:02.760871+00:00', 'best_validation_auc': 0.9806138933764135, 'best_epoch': 9, 'seed': 42, 'training_sources': ['RFMiD'], 'clinical_status': 'NOT CLINICALLY VALIDATED', 'warning': 'ODE is not synonymous with papilledema. Research-only; not for diagnosis or care decisions.'}
Are model weights identical? True


In [20]:
print('g1')

g1
